# Credit Card Fraud Detection – Support Vector Machine (SVM)
**Member 4**  
**Algorithm:** Support Vector Machine  
**Dataset:** [Kaggle – Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

SVM finds the optimal hyperplane that maximises the margin between classes. With the RBF kernel it can model non-linear boundaries. We use a subset of the data for SVM due to its O(n²) scaling.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, f1_score
)

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Preprocessed Data

In [ ]:
X_train_full, X_test, y_train_full, y_test = joblib.load('preprocessed_data.pkl')
print(f'Train: {X_train_full.shape}, Test: {X_test.shape}')

# SVM is computationally expensive – use a stratified subsample for training
from sklearn.model_selection import train_test_split
X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full,
    train_size=0.3, random_state=42, stratify=y_train_full
)
print(f'SVM training subset: {X_train.shape}')

## 2. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

svm_base = SVC(probability=True, class_weight='balanced', random_state=42)
grid_search = GridSearchCV(svm_base, param_grid, cv=3,
                           scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
svm = grid_search.best_estimator_

## 3. Predictions & Evaluation

In [ ]:
y_pred = svm.predict(X_test)
y_prob = svm.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print(f'Avg Prec : {average_precision_score(y_test, y_prob):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')

## 4. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Legitimate','Fraud'], yticklabels=['Legitimate','Fraud'])
axes[0].set_title('Confusion Matrix – SVM')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='purple', lw=2, label=f'AUC = {auc_score:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_title('ROC Curve – SVM')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[2].plot(rec, prec, color='indigo', lw=2, label=f'AP = {ap:.4f}')
axes[2].set_title('Precision-Recall – SVM')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].legend()

plt.tight_layout()
plt.savefig('svm_results.png', dpi=150)
plt.show()

## 5. Algorithm Comparison Across All 4 Models

In [ ]:
# Fill in these values after running all 4 notebooks
comparison = pd.DataFrame({
    'Algorithm': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM'],
    'ROC-AUC':   [0.000, 0.000, 0.000, roc_auc_score(y_test, y_prob)],
    'Avg Prec':  [0.000, 0.000, 0.000, average_precision_score(y_test, y_prob)],
    'F1-Score':  [0.000, 0.000, 0.000, f1_score(y_test, y_pred)]
})

# Update the first 3 rows manually after collecting results
print(comparison.to_string(index=False))

In [ ]:
# Bar chart comparison (update values above first)
metrics = ['ROC-AUC', 'Avg Prec', 'F1-Score']
x = np.arange(len(comparison))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['steelblue', 'green', 'darkorange', 'purple']

for i, metric in enumerate(metrics):
    ax.bar(x + i * width, comparison[metric], width, label=metric)

ax.set_xticks(x + width)
ax.set_xticklabels(comparison['Algorithm'], rotation=15)
ax.set_ylim(0, 1.05)
ax.set_title('Algorithm Comparison – All Metrics')
ax.legend()
plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150)
plt.show()

## 6. Summary
| Metric | Value |
|--------|-------|
| ROC-AUC | See output above |
| Average Precision | See output above |
| F1-Score (Fraud) | See output above |

**Observations:**
- SVM with RBF kernel can find non-linear decision boundaries in high-dimensional PCA-transformed space.
- Training on a 30% subset is a necessary trade-off due to SVM's O(n²) complexity.
- `probability=True` enables calibrated probabilities for ROC/PR curves.
- Limitation: does not scale well to very large datasets; slower than tree-based methods.